# 02 — Preprocesamiento (§6 del README)

| | |
|---|---|
| **Entrada** | `data/raw/bank-additional-full.csv` (41.188 filas × 21 columnas) |
| **Evidencia previa** | `notebooks/01_exploratory_analysis.ipynb` (README §5) |
| **Escenario** | Predicción **antes de la llamada** → 15 variables disponibles (README §3) |
| **Salida** | `X_train`, `X_test`, `y_train`, `y_test` + preprocesador reproducible |
| **Código reutilizable** | `src/preprocessing.py` |

**Reglas (README):** cada decisión se justifica con evidencia y el preprocesador se ajusta **solo con datos de entrenamiento** para evitar fuga de información.

## 1. Carga y eliminación de duplicados

Se carga el CSV original con `sep=';'` (ver notebook 01, §1) y se eliminan las **12 filas duplicadas exactas** detectadas en el EDA.

- Los duplicados no aportan información nueva (son idénticos en las 21 columnas) y, si una copia cae en train y otra en test, inflan artificialmente las métricas.
- La eliminación se hace **antes del split**.
- No se guarda una copia en `data/processed/`: el pipeline reproduce la limpieza y el archivo original permanece intacto en `data/raw/` (README §15).

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd

from src.preprocessing import (
    CATEGORICAL_FEATURES,
    DROPPED_FEATURES,
    EXCLUDED_FEATURES,
    FEATURES,
    NUMERIC_FEATURES,
    RANDOM_STATE,
    TEST_SIZE,
    build_preprocessor,
    clean_data,
    load_raw_data,
    split_features_target,
    train_test_split_data,
)

df_raw = load_raw_data()
n_duplicados = int(df_raw.duplicated().sum())
df = clean_data(df_raw)

print(f"Original: {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas")
print(f"Duplicados exactos eliminados: {n_duplicados}")
print(f"Limpio: {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"Celdas vacias en el original: {int(df_raw.isna().sum().sum())}")

Original: 41,188 filas x 21 columnas
Duplicados exactos eliminados: 12
Limpio: 41,176 filas x 21 columnas
Celdas vacias en el original: 0


## 2. Separación de características y objetivo

Del EDA y del escenario fijado (README §3) se derivan estas decisiones:

| Decisión | Variables | Evidencia |
|---|---|---|
| **Fuera por disponibilidad** | `duration`, `campaign`, `month`, `day_of_week`, `contact` | No existen antes de la llamada (README §3; notebook 01 §10) |
| **Fuera por centinela no fiable** | `pdays` | 39.673 de 41.188 valores son `999` (96,3%); 4.110 filas con `previous > 0` lo contradicen; solo 1.515 tienen un valor real |
| **Fuera por baja información** | `default` | Solo 3 `yes` en 41.188 filas (0,007%) y 20,9% `unknown` |
| **Se conserva** | `unknown` en el resto de categóricas | Missing codificado como categoría (ficha, línea 72): no se imputa, se codifica como categoría propia |

**¿Y el indicador `contactado_antes`?** El EDA verificó que `previous = 0` ↔ `poutcome = nonexistent` coinciden al 100%. Un indicador derivado de `previous > 0` sería **exactamente redundante** con `poutcome`, por lo que se omite para no duplicar información.

**Set final: 13 features** (7 numéricas + 6 categóricas):

- Numéricas: `age`, `previous`, `emp.var.rate`, `cons.price.idx`, `cons.conf.idx`, `euribor3m`, `nr.employed`.
- Categóricas: `job`, `marital`, `education`, `housing`, `loan`, `poutcome`.

El objetivo `y` se convierte a binario: `yes = 1`, `no = 0`.

In [2]:
X, y = split_features_target(df)

print(f"X: {X.shape[0]:,} filas x {X.shape[1]} features")
print(f"Numericas ({len(NUMERIC_FEATURES)}): {NUMERIC_FEATURES}")
print(f"Categoricas ({len(CATEGORICAL_FEATURES)}): {CATEGORICAL_FEATURES}")

print("\nObjetivo y_bin:")
print(y.value_counts().rename(index={0: "no (0)", 1: "yes (1)"}).to_string())
print("\nProporcion (%):")
print((y.value_counts(normalize=True).rename(index={0: "no", 1: "yes"}) * 100).round(2).to_string())

assert not set(EXCLUDED_FEATURES + DROPPED_FEATURES) & set(X.columns)
print("\nVerificacion: ninguna variable excluida esta en X.")

X: 41,176 filas x 13 features
Numericas (7): ['age', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
Categoricas (6): ['job', 'marital', 'education', 'housing', 'loan', 'poutcome']

Objetivo y_bin:
y_bin
no (0)     36537
yes (1)     4639

Proporcion (%):
y_bin
no     88.73
yes    11.27

Verificacion: ninguna variable excluida esta en X.


## 3. División train/test

- **80% train / 20% test**, con `random_state=42` para reproducibilidad.
- **Estratificada por `y`**: mantiene la proporción 88,7% / 11,3% en ambas particiones. Sin estratificar, una partición podría quedar con muy pocos `yes` y las métricas serían inestables.
- El split se hace **antes de cualquier ajuste** (escalado, One-Hot): así el test nunca participa del cálculo de medias, desvíos ni categorías.

In [3]:
X_train, X_test, y_train, y_test = train_test_split_data(X, y)

print(f"Train: {X_train.shape[0]:,} filas ({X_train.shape[0] / len(X) * 100:.1f}%)")
print(f"Test : {X_test.shape[0]:,} filas ({X_test.shape[0] / len(X) * 100:.1f}%)")
print("\nProporcion de clases (train vs test):")
print(pd.DataFrame({
    "train_%": (y_train.value_counts(normalize=True) * 100).round(2),
    "test_%": (y_test.value_counts(normalize=True) * 100).round(2),
}).rename(index={0: "no", 1: "yes"}).to_string())

Train: 32,940 filas (80.0%)
Test : 8,236 filas (20.0%)

Proporcion de clases (train vs test):
       train_%  test_%
y_bin                 
no       88.73   88.73
yes      11.27   11.27


## 4. Tratamiento de numéricas y categóricas

| Grupo | Transformación | Por qué |
|---|---|---|
| Numéricas | `StandardScaler` | Media 0 y desvío 1: necesario para modelos lineales regularizados; no altera el orden ni la información |
| Categóricas | `OneHotEncoder(handle_unknown='ignore')` | Convierte categorías en columnas 0/1 sin imponer un orden falso. `handle_unknown='ignore'` evita errores si en test aparece una categoría no vista en train |
| `unknown` | Se mantiene como categoría | Decisión del EDA: no se inventan datos |
| Atípicos | **No se eliminan** | El EDA mostró que no son errores (p. ej. `previous=7`); eliminarlos sesgaría la muestra. Los modelos de árbol serán naturalmente robustos y los lineales se evalúan con regularización |
| Desbalance | Se evalúa en §7 con `class_weight` | No se remuestrea ni se sintetizan datos sin evidencia previa |

Todo se encapsula en un **`ColumnTransformer`** dentro de un **`Pipeline`**: el ajuste ocurre con train y la misma transformación se aplica a test.

## 5. Pipeline de preprocesamiento

Se ajusta el `ColumnTransformer` **solo con `X_train`** y se inspecciona la salida. Este ajuste es solo demostrativo: en el notebook 03 (y en toda evaluación posterior) el preprocesador viaja **dentro del `Pipeline` del modelo**, de modo que en validación cruzada se re-ajuste en cada partición y no haya fuga entre folds.

In [4]:
preprocessor = build_preprocessor()
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

nombres = preprocessor.get_feature_names_out()

print(f"X_train transformado: {X_train_prep.shape}")
print(f"X_test transformado : {X_test_prep.shape}")
print(f"Columnas generadas  : {len(nombres)}")
print("\nPrimeras 20 columnas generadas:")
print(list(nombres[:20]))

pd.DataFrame(X_train_prep[:5], columns=nombres).iloc[:, :8].round(2)

X_train transformado: (32940, 40)
X_test transformado : (8236, 40)
Columnas generadas  : 40

Primeras 20 columnas generadas:
['age', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'job_admin.', 'job_blue-collar', 'job_entrepreneur', 'job_housemaid', 'job_management', 'job_retired', 'job_self-employed', 'job_services', 'job_student', 'job_technician', 'job_unemployed', 'job_unknown', 'marital_divorced']


,age,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,job_admin.
0,-1.53,1.66,-1.90,-1.06,-0.06,-1.36,-1.25,0.0
1,-0.77,-0.35,0.65,0.72,0.89,0.71,0.33,0.0
2,-0.67,-0.35,-0.11,-0.65,-0.32,0.36,0.40,0.0
3,-0.19,-0.35,0.84,0.59,-0.47,0.77,0.85,0.0
4,-0.10,-0.35,0.84,0.59,-0.47,0.77,0.85,0.0


## 6. Verificación anti-fuga

Comprobaciones realizadas:

1. **Split antes del ajuste**: `X_test` no participa en `fit_transform`.
2. **Escalador ajustado solo con train**: sus medias deben coincidir con las de `X_train` y no exactamente con las del dataset completo.
3. **Sin variables fuera del escenario**: ninguna columna de `EXCLUDED_FEATURES` ni `DROPPED_FEATURES` está en `X`.
4. **Sin codificación basada en el objetivo**: no se usan medias de `y` por categoría (target encoding).
5. **Duplicados eliminados antes del split**: evita copias idénticas en train y test.

In [5]:
scaler = preprocessor.named_transformers_["num"].named_steps["scaler"]

comparacion = pd.DataFrame({
    "media_train (usada)": scaler.mean_,
    "media_dataset_completo (NO usada)": X[NUMERIC_FEATURES].mean().values,
}, index=NUMERIC_FEATURES).round(3)

print("Medias del escalador vs medias del dataset completo:")
print(comparacion.to_string())
print("\nDiferencia maxima:", (comparacion.iloc[:, 0] - comparacion.iloc[:, 1]).abs().max().round(3))
print("-> El escalador se ajusto solo con train (las medias no coinciden exactamente).")

Medias del escalador vs medias del dataset completo:
                media_train (usada)  media_dataset_completo (NO usada)
age                          40.017                             40.024
previous                      0.174                              0.173
emp.var.rate                  0.080                              0.082
cons.price.idx               93.576                             93.576
cons.conf.idx               -40.510                            -40.503
euribor3m                     3.618                              3.621
nr.employed                5166.906                           5167.035

Diferencia maxima: 0.129
-> El escalador se ajusto solo con train (las medias no coinciden exactamente).


## 7. Resumen de §6 (checklist del README)

| Punto del README §6 | Decisión / evidencia |
|---|---|
| Separación características/objetivo | `X`: 13 features · `y`: binaria (`yes=1`) |
| División train/test | 80/20 · `random_state=42` |
| Estratificación | Sí, por `y` (11,3% de `yes` en ambas particiones) |
| Numéricas | `StandardScaler` |
| Categóricas | One-Hot (`unknown` como categoría; `handle_unknown='ignore'`) |
| Codificación y escalado | Dentro de `ColumnTransformer` + `Pipeline` |
| Desbalance | Opciones documentadas; comparación empírica en §7 |
| `Pipeline` / `ColumnTransformer` | Sí — ver `src/preprocessing.py` |
| Prevención de fuga | Split previo al ajuste; preprocesador solo con train; sin variables no disponibles |

**Datos**: 41.176 filas tras eliminar duplicados (41.188 − 12).

## 8. Próximo paso

§7 — Baseline: referencia trivial (`DummyClassifier`) y regresión logística como modelo simple e interpretable, usando este mismo split y pipeline.